In [ ]:
"""
=============================================================
FILE 41 — SUPERVISOR WORKER PATTERN
=============================================================

CONCEPTS TAUGHT
----------------
1. Supervisor Worker Architecture
2. Centralized Control
3. Worker Coordination
4. Enterprise Orchestration
5. Task Delegation
6. AI Supervision
7. Controlled Execution
8. Multi-Agent Governance
9. Centralized Oversight
10. AI Team Management

CORE IDEA
-----------
A supervisor agent:
- delegates tasks
- monitors workers
- validates outputs
- creates final answer

FLOW
-----
Supervisor
   ↓
Worker Agents
   ↓
Supervisor Validation
   ↓
Final Output

REAL WORLD USE CASES
---------------------
- Enterprise AI operations
- AI governance systems
- Managed automation
- AI orchestration platforms
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display

# ============================================================
# STEP 2 — LOAD ENV
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — LLM
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — STATE
# ============================================================

class State(TypedDict):

    task: str

    research_output: str
    analytics_output: str

    supervisor_review: str

# ============================================================
# STEP 5 — WORKER AGENTS
# ============================================================

def research_worker(state: State):

    response = llm.invoke(
        f"""
        Perform market research for:

        {state['task']}
        """
    )

    return {
        "research_output": response.content
    }

def analytics_worker(state: State):

    response = llm.invoke(
        f"""
        Perform business analytics for:

        {state['task']}
        """
    )

    return {
        "analytics_output": response.content
    }

# ============================================================
# STEP 6 — SUPERVISOR AGENT
# ============================================================

def supervisor_agent(state: State):

    response = llm.invoke(
        f"""
        Review outputs from all workers.

        RESEARCH OUTPUT:
        {state['research_output']}

        ANALYTICS OUTPUT:
        {state['analytics_output']}

        Create:
        - executive summary
        - validation
        - final recommendation
        """
    )

    return {
        "supervisor_review": response.content
    }

# ============================================================
# STEP 7 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node("research_worker", research_worker)
builder.add_node("analytics_worker", analytics_worker)

builder.add_node("supervisor_agent", supervisor_agent)

# ============================================================
# STEP 8 — PARALLEL EXECUTION
# ============================================================

builder.add_edge(START, "research_worker")
builder.add_edge(START, "analytics_worker")

builder.add_edge(
    "research_worker",
    "supervisor_agent"
)

builder.add_edge(
    "analytics_worker",
    "supervisor_agent"
)

builder.add_edge(
    "supervisor_agent",
    END
)

# ============================================================
# STEP 9 — COMPILE
# ============================================================

graph = builder.compile()

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

# ============================================================
# STEP 10 — RUN WORKFLOW
# ============================================================

result = graph.invoke(
    {
        "task":
        """
        AI adoption roadmap for an insurance company
        """
    }
)

# ============================================================
# STEP 11 — PRINT RESULT
# ============================================================

print("\nSUPERVISOR FINAL REVIEW\n")
print("=" * 60)

print(result["supervisor_review"])